# Padel Game Analytics — Exploration & Dashboard

This notebook lets you:
1. Inspect the raw input video frame by frame
2. Run the full pipeline on a short clip
3. Explore shot events interactively
4. Render the analytics dashboard

**Run cells top-to-bottom. All outputs are saved to `data/outputs/`.**

## 0 · Setup

In [3]:
# ── Standard library ─────────────────────────────────────────────────────────
import sys
import json
from pathlib import Path

# ── Add src/ to path ──────────────────────────────────────────────────────────
ROOT = Path("../").resolve()
SRC  = ROOT / "src"
sys.path.insert(0, str(SRC))

print(f"Project root : {ROOT}")
print(f"src/         : {SRC}")
print("Path configured ✓")

Project root : C:\Users\javed\Desktop\Panel Analytics\padel-analytics
src/         : C:\Users\javed\Desktop\Panel Analytics\padel-analytics\src
Path configured ✓


In [4]:
# ── Third-party ───────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Image as IPImage
from tqdm.notebook import tqdm

matplotlib.rcParams["figure.dpi"] = 120
plt.style.use("dark_background")

print("Imports OK ✓")
print(f"  OpenCV    {cv2.__version__}")
print(f"  NumPy     {np.__version__}")
print(f"  Pandas    {pd.__version__}")

Imports OK ✓
  OpenCV    4.11.0
  NumPy     1.26.4
  Pandas    2.2.2


In [5]:
# ── Project modules ───────────────────────────────────────────────────────────
from config          import *
from utils           import get_video_info, validate_video, frame_to_timestamp
from detection       import Detector, DetectionResult
from tracking        import Tracker
from pose_estimator  import PoseEstimator
from shot_classifier import ShotClassifier, ShotType
from analytics       import Analytics
from visualizer      import Visualizer
from exporter        import Exporter

print("Project modules loaded ✓")

ImportError: cannot import name 'LOG_LEVEL' from 'config' (C:\Users\javed\Desktop\Panel Analytics\padel-analytics\src\config.py)

## 1 · Video Inspection

In [ ]:
# ── Point this at your video ───────────────────────────────────────────────────
VIDEO_PATH = ROOT / "data" / "raw" / "match.mp4"

ok, msg = validate_video(VIDEO_PATH)
print(f"Video valid : {ok}  ({msg})")

if ok:
    info = get_video_info(VIDEO_PATH)
    print("\nVideo metadata:")
    for k, v in info.items():
        print(f"  {k:<15} : {v}")

In [ ]:
def show_frames(video_path, frame_indices, cols=3, figsize=(15, 5)):
    """
    Display a grid of frames from the video at the given frame indices.
    """
    cap  = cv2.VideoCapture(str(video_path))
    fps  = cap.get(cv2.CAP_PROP_FPS)
    rows = (len(frame_indices) + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()

    for ax, fidx in zip(axes, frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
        ret, frame = cap.read()
        if ret:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            ax.imshow(rgb)
            ax.set_title(
                f"Frame {fidx}  |  {frame_to_timestamp(fidx, fps)}",
                fontsize=9, color="white",
            )
        ax.axis("off")

    for ax in axes[len(frame_indices):]:
        ax.axis("off")

    plt.suptitle("Sample Frames", color="white", fontsize=13)
    plt.tight_layout()
    plt.show()
    cap.release()


# Show frames at 0%, 10%, 25%, 50%, 75%, 90% of the video
if ok:
    total = info["frame_count"]
    sample_indices = [
        int(total * pct)
        for pct in [0.0, 0.10, 0.25, 0.50, 0.75, 0.90]
    ]
    show_frames(VIDEO_PATH, sample_indices)

## 2 · Detection Preview
Run YOLO on a handful of frames and display bounding boxes.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
PREVIEW_FRAMES = [0, 30, 60, 120, 200, 300]   # frame indices to preview
CONF_THRESH    = 0.40
DEVICE         = "cpu"                          # change to '0' for GPU

detector = Detector(conf=CONF_THRESH, device=DEVICE)
print(f"Detector ready: {detector}")

In [ ]:
def draw_detections(frame, det_result):
    """Draw detection boxes on a copy of the frame and return RGB."""
    canvas = frame.copy()
    for det in det_result.players:
        x1, y1, x2, y2 = det.bbox
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 200, 100), 2)
        cv2.putText(
            canvas, f"P {det.confidence:.0%}",
            (x1, max(0, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
            (0, 200, 100), 2, cv2.LINE_AA,
        )
    for ball in det_result.balls:
        x1, y1, x2, y2 = ball.bbox
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        cv2.circle(canvas, (cx, cy), 10, (0, 165, 255), 2)
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


cap  = cv2.VideoCapture(str(VIDEO_PATH))
cols = 3
rows = (len(PREVIEW_FRAMES) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
axes = np.array(axes).flatten()

for ax, fidx in zip(axes, PREVIEW_FRAMES):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ret, frame = cap.read()
    if ret:
        det_result = detector.detect(frame, frame_idx=fidx)
        rgb        = draw_detections(frame, det_result)
        ax.imshow(rgb)
        ax.set_title(
            f"Frame {fidx} | "
            f"Players={det_result.player_count} "
            f"Ball={det_result.has_ball}",
            fontsize=9, color="white",
        )
    ax.axis("off")

for ax in axes[len(PREVIEW_FRAMES):]:
    ax.axis("off")

cap.release()
plt.suptitle("Detection Preview (YOLOv8)",
             color="white", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 3 · Run the Full Pipeline on a Short Clip

Adjust `MAX_FRAMES` to control how many frames to process.
Recommended: `150–300` frames for a quick test.

In [ ]:
MAX_FRAMES = 300    # ← change this to process more of the video
FPS        = info["fps"]
FRAME_W    = info["width"]
FRAME_H    = info["height"]

# ── Reinitialise all modules cleanly ─────────────────────────────────────────
tracker    = Tracker()
classifier = ShotClassifier(velocity_threshold=SHOT_VELOCITY_THRESHOLD)
analytics  = Analytics(frame_w=FRAME_W, frame_h=FRAME_H)
visualizer = Visualizer(frame_w=FRAME_W, frame_h=FRAME_H,
                        show_skeleton=True, show_hud=True)
exporter   = Exporter(output_dir=ROOT / "data" / "outputs")

try:
    pose_estimator = PoseEstimator()
    print("PoseEstimator ready ✓")
except Exception as e:
    pose_estimator = None
    print(f"PoseEstimator unavailable: {e}")

print(f"Running pipeline on first {MAX_FRAMES} frames...")

In [ ]:
# ── Open output video writer ──────────────────────────────────────────────────
nb_video_out = ROOT / "data" / "outputs" / "notebook_output.mp4"
exporter.open_video_writer(FRAME_W, FRAME_H, FPS, path=nb_video_out)

# ── Pipeline loop ─────────────────────────────────────────────────────────────
cap       = cv2.VideoCapture(str(VIDEO_PATH))
frame_idx = 0

# Store a few annotated frames for display
sample_annotated = []
SAMPLE_AT = {30, 60, 100, 150, 200, 250}

for _ in tqdm(range(MAX_FRAMES), desc="Pipeline", unit="frame"):
    ret, frame = cap.read()
    if not ret:
        break

    # 1. Detect
    det_result = detector.detect(frame, frame_idx=frame_idx)

    # 2. Track
    tracks = tracker.update(det_result, frame)

    # 3. Pose
    poses = []
    if pose_estimator and tracks:
        poses = pose_estimator.estimate(frame, tracks)

    # 4. Classify
    new_events = []
    if poses:
        classifier.feed_batch(poses, frame_idx=frame_idx, fps=FPS)
        new_events = classifier.get_new_events()

    # 5. Analytics
    analytics.update(
        new_events=new_events,
        tracks=tracks,
        frame_idx=frame_idx,
        fps=FPS,
    )

    # 6. Visualize
    annotated = visualizer.draw(
        frame=frame,
        det_result=det_result,
        tracks=tracks,
        poses=poses,
        new_events=new_events,
        shot_counts=analytics.shot_counts(),
        frame_idx=frame_idx,
        fps=FPS,
    )

    # 7. Write
    exporter.write_frame(annotated)

    # Save sample frames for display below
    if frame_idx in SAMPLE_AT:
        sample_annotated.append(
            (frame_idx, cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        )

    frame_idx += 1

cap.release()
exporter.close_video_writer()

print(f"\nPipeline complete.")
print(f"  Frames processed : {frame_idx}")
print(f"  Total shots      : {analytics.total_shots()}")
print(f"  Rallies detected : {len(analytics.rallies())}")
print(f"  Output video     : {nb_video_out}")

---
## 4 · Annotated Frame Preview

In [ ]:
if sample_annotated:
    cols = min(3, len(sample_annotated))
    rows = (len(sample_annotated) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
    axes = np.array(axes).flatten()

    for ax, (fidx, rgb) in zip(axes, sample_annotated):
        ax.imshow(rgb)
        ax.set_title(
            f"Frame {fidx} — annotated",
            fontsize=9, color="white",
        )
        ax.axis("off")

    for ax in axes[len(sample_annotated):]:
        ax.axis("off")

    plt.suptitle("Annotated Output Frames",
                 color="white", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No sample frames collected — adjust SAMPLE_AT set above.")

## 5 · Shot Events Table

In [ ]:
all_events = classifier.all_events

if all_events:
    df = pd.DataFrame([e.to_dict() for e in all_events])
    df = df.sort_values("frame_idx").reset_index(drop=True)

    print(f"Total shot events: {len(df)}")
    print(f"Shot type counts:\n{df['shot_type'].value_counts().to_string()}")
    print()
    display(df.head(20))
else:
    print("No shot events detected yet.")
    print("Try increasing MAX_FRAMES or lowering --vel-threshold.")
    df = pd.DataFrame()

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
if not df.empty:
    print("Numerical summary:")
    display(
        df[["wrist_velocity", "forearm_angle",
            "confidence", "timestamp_sec"]].describe().round(3)
    )

---
## 6 · Shot Distribution Charts

In [ ]:
fig = analytics.plot_shot_counts(
    save_path=str(ROOT / "data" / "outputs" / "chart_shot_counts.png")
)
if fig:
    plt.show()
else:
    print("No shot data to plot yet.")

In [ ]:
fig = analytics.plot_shot_timeline(
    save_path=str(ROOT / "data" / "outputs" / "chart_timeline.png")
)
if fig:
    plt.show()
else:
    print("No shot events to plot yet.")

---
## 7 · Court Position Heatmap

In [ ]:
# All players combined
fig = analytics.plot_heatmap(
    track_id=None,
    save_path=str(ROOT / "data" / "outputs" / "chart_heatmap_all.png"),
)
if fig:
    plt.show()
else:
    print("No position data yet.")

In [ ]:
# Per-player heatmaps
counts = analytics.shot_counts()
for tid in sorted(counts.keys()):
    fig = analytics.plot_heatmap(
        track_id=tid,
        save_path=str(
            ROOT / "data" / "outputs" / f"chart_heatmap_player{tid}.png"
        ),
    )
    if fig:
        plt.show()

---
## 8 · Rally Analysis

In [ ]:
rallies = analytics.rallies()

if rallies:
    rally_df = pd.DataFrame([r.to_dict() for r in rallies])
    print(f"Total rallies detected: {len(rallies)}")
    display(rally_df)

    # ── Rally duration distribution ───────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.patch.set_facecolor("#1a1a2e")

    for ax in axes:
        ax.set_facecolor("#16213e")
        ax.tick_params(colors="white")
        ax.spines[:].set_color("#ffffff33")

    # Duration histogram
    axes[0].hist(
        rally_df["duration_sec"],
        bins=max(5, len(rallies) // 3),
        color="#2196F3", edgecolor="#ffffff22",
    )
    axes[0].set_xlabel("Duration (seconds)", color="white")
    axes[0].set_ylabel("Rally count",        color="white")
    axes[0].set_title("Rally Duration Distribution",
                      color="white", fontsize=11)

    # Shot count per rally
    axes[1].bar(
        rally_df["rally_id"],
        rally_df["shot_count"],
        color="#FF9800", edgecolor="#ffffff22",
    )
    axes[1].set_xlabel("Rally ID",    color="white")
    axes[1].set_ylabel("Shot count",  color="white")
    axes[1].set_title("Shots per Rally",
                      color="white", fontsize=11)

    plt.tight_layout()
    plt.savefig(
        str(ROOT / "data" / "outputs" / "chart_rallies.png"),
        dpi=150, bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.show()
else:
    print("No rallies detected yet. Try processing more frames.")

---
## 9 · Velocity & Angle Distributions

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.patch.set_facecolor("#1a1a2e")

    SHOT_COLOURS_HEX = {
        ShotType.FOREHAND: "#2196F3",
        ShotType.BACKHAND: "#F44336",
        ShotType.SMASH:    "#FF9800",
        ShotType.UNKNOWN:  "#9E9E9E",
    }

    for ax in axes:
        ax.set_facecolor("#16213e")
        ax.tick_params(colors="white")
        ax.spines[:].set_color("#ffffff33")

    # ── Wrist velocity by shot type ───────────────────────────────────────
    for stype, grp in df.groupby("shot_type"):
        axes[0].hist(
            grp["wrist_velocity"],
            bins=15,
            alpha=0.7,
            label=stype.capitalize(),
            color=SHOT_COLOURS_HEX.get(stype, "grey"),
            edgecolor="#ffffff22",
        )
    axes[0].set_xlabel("Wrist velocity (px/frame)", color="white")
    axes[0].set_ylabel("Count", color="white")
    axes[0].set_title("Velocity Distribution", color="white", fontsize=11)
    axes[0].legend(facecolor="#0f3460", labelcolor="white", fontsize=8)

    # ── Forearm angle by shot type ────────────────────────────────────────
    for stype, grp in df.groupby("shot_type"):
        axes[1].hist(
            grp["forearm_angle"],
            bins=15,
            alpha=0.7,
            label=stype.capitalize(),
            color=SHOT_COLOURS_HEX.get(stype, "grey"),
            edgecolor="#ffffff22",
        )
    axes[1].axvline(0, color="white", linestyle="--",
                    linewidth=1, alpha=0.5, label="Vertical")
    axes[1].set_xlabel("Forearm angle (degrees)", color="white")
    axes[1].set_ylabel("Count", color="white")
    axes[1].set_title("Angle Distribution", color="white", fontsize=11)
    axes[1].legend(facecolor="#0f3460", labelcolor="white", fontsize=8)

    # ── Confidence scores ─────────────────────────────────────────────────
    for stype, grp in df.groupby("shot_type"):
        axes[2].hist(
            grp["confidence"],
            bins=10,
            alpha=0.7,
            label=stype.capitalize(),
            color=SHOT_COLOURS_HEX.get(stype, "grey"),
            edgecolor="#ffffff22",
        )
    axes[2].set_xlabel("Confidence score", color="white")
    axes[2].set_ylabel("Count", color="white")
    axes[2].set_title("Confidence Distribution", color="white", fontsize=11)
    axes[2].legend(facecolor="#0f3460", labelcolor="white", fontsize=8)

    plt.tight_layout()
    plt.savefig(
        str(ROOT / "data" / "outputs" / "chart_distributions.png"),
        dpi=150, bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    plt.show()
else:
    print("No event data to plot.")

---
## 10 · Full Analytics Dashboard

In [ ]:
fig = analytics.plot_dashboard(
    save_path=str(ROOT / "data" / "outputs" / "chart_dashboard.png")
)
if fig:
    plt.show()
else:
    print("Not enough data for dashboard yet.")

---
## 11 · Export All Results

In [ ]:
all_events    = classifier.all_events
match_summary = analytics.match_summary()

written = exporter.export_all(
    shot_events=all_events,
    match_summary=match_summary,
    analytics=analytics,
)

print("\nFiles written:")
for name, path in written.items():
    print(f"  {name:<15} → {path}")

---
## 12 · Inspect shots.json

In [ ]:
json_path = ROOT / "data" / "outputs" / "shots.json"

if json_path.exists():
    with open(json_path) as f:
        data = json.load(f)

    print("── meta ──────────────────────────────────")
    for k, v in data["meta"].items():
        print(f"  {k:<18}: {v}")

    print("\n── summary ───────────────────────────────")
    summary = data["summary"]
    print(f"  Duration      : {summary.get('duration_sec')}s")
    print(f"  Total shots   : {summary.get('total_shots')}")
    print(f"  Total rallies : {summary.get('total_rallies')}")

    print("\n── first 3 shots ─────────────────────────")
    for shot in data["shots"][:3]:
        print(f"  {shot}")
else:
    print("shots.json not found — run the export cell above first.")

---
## 13 · Match Summary Print

In [ ]:
counts  = analytics.shot_counts()
rallies = analytics.rallies()

print("╔══════════════════════════════════════════╗")
print("║     PADEL ANALYTICS — MATCH SUMMARY      ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Total shots   : {analytics.total_shots():<23}║")
print(f"║  Total rallies : {len(rallies):<23}║")

if rallies:
    avg_dur = sum(r.duration_sec for r in rallies) / len(rallies)
    print(f"║  Avg rally dur : {avg_dur:<.1f}s{'':<21}║")

print("╠══════════════════════════════════════════╣")

for tid in sorted(counts.keys()):
    c    = counts[tid]
    fh   = c.get("forehand", 0)
    bh   = c.get("backhand", 0)
    sm   = c.get("smash",    0)
    tot  = c.get("total",    0)
    rate = analytics.shot_rate_per_minute(tid)
    print(f"║  Player {tid}  Total={tot}  FH={fh}  BH={bh}  SM={sm}   ║")
    print(f"║           Rate : {rate} shots/min{'':<11}║")

print("╚══════════════════════════════════════════╝")